In [53]:
import pandas as pd
import numpy as np
import pickle

In [54]:
df = pd.read_csv("Instagram_Analytics.csv")
df.head()

,post_id,account_id,account_type,follower_count,media_type,content_category,traffic_source,has_call_to_action,post_datetime,post_date,...,comments,shares,saves,reach,impressions,engagement_rate,followers_gained,caption_length,hashtags_count,performance_bucket_label
0,IG0000001,7,brand,3551,reel,Technology,Home Feed,1,2024-11-30 06:00:00,2024-11-30,...,5,7,34,4327,6230,0.0385,899,100,7,medium
1,IG0000002,20,creator,31095,image,Fitness,Hashtags,1,2025-08-15 15:00:00,2025-08-15,...,10,21,68,7451,8268,0.0663,805,122,5,viral
2,IG0000003,15,brand,8167,reel,Beauty,Reels Feed,0,2025-09-11 16:00:00,2025-09-11,...,2,1,22,1639,2616,0.0531,758,115,8,high
3,IG0000004,11,creator,9044,carousel,Music,External,0,2025-09-18 03:00:00,2025-09-18,...,0,7,0,2877,3171,0.0309,402,115,7,medium
4,IG0000005,8,creator,15986,reel,Technology,Profile,0,2025-03-21 09:00:00,2025-03-21,...,8,5,21,5350,8503,0.0221,155,112,9,low


In [55]:
df = df.drop([
    "post_id",
    "account_id",
    "post_datetime",
    "post_date",
    "traffic_source"
], axis=1)

In [56]:
# Engagement features
df["engagement_score"] = df["comments"] + df["shares"] + df["saves"]

df["weighted_engagement"] = (
    df["comments"] * 1 +
    df["shares"] * 2 +
    df["saves"] * 3
)

df["engagement_per_follower"] = df["weighted_engagement"] / (df["follower_count"] + 1)

df["log_followers"] = np.log1p(df["follower_count"])

df["caption_efficiency"] = df["caption_length"] / (df["hashtags_count"] + 1)

df["is_peak"] = df["post_hour"].apply(lambda x: 1 if 18 <= x <= 22 else 0)

df["is_weekend"] = df["day_of_week"].apply(lambda x: 1 if x in ["Saturday", "Sunday"] else 0)

# 🔥 NEW FEATURES (for medium detection)
df["caption_per_follower"] = df["caption_length"] / (df["log_followers"] + 1)

df["balanced_score"] = (
    df["caption_length"] * 0.3 +
    df["hashtags_count"] * 2 +
    df["is_peak"] * 5
)

df["hashtag_efficiency"] = df["hashtags_count"] / (df["caption_length"] + 1)

In [57]:
df = df.drop([
    "comments",
    "shares",
    "saves",
    "reach",
    "impressions",
    "engagement_rate",
    "follower_count"
], axis=1)

In [58]:
def simplify(label):
    if label == "low":
        return "low"
    elif label == "medium":
        return "medium"
    else:
        return "high"

df["performance_simple"] = df["performance_bucket_label"].apply(simplify)

In [59]:
y = df["performance_simple"]
X = df.drop(["performance_bucket_label", "performance_simple", "likes"], axis=1)

In [60]:
X = pd.get_dummies(X, drop_first=True)
print("Shape:", X.shape)

Shape: (29999, 33)


In [61]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [62]:
from sklearn.utils import resample

train_df = X_train.copy()
train_df["target"] = y_train

low = train_df[train_df["target"] == "low"]
medium = train_df[train_df["target"] == "medium"]
high = train_df[train_df["target"] == "high"]

max_size = max(len(low), len(medium), len(high))

low = resample(low, replace=True, n_samples=max_size, random_state=42)
medium = resample(medium, replace=True, n_samples=max_size, random_state=42)
high = resample(high, replace=True, n_samples=max_size, random_state=42)

train_balanced = pd.concat([low, medium, high])

X_train = train_balanced.drop("target", axis=1)
y_train = train_balanced["target"]

In [63]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

,n_estimators,300
,criterion,'gini'
,max_depth,20
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [64]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.5675
              precision    recall  f1-score   support

        high       0.69      0.70      0.70      3000
         low       0.51      0.62      0.56      1500
      medium       0.33      0.25      0.28      1500

    accuracy                           0.57      6000
   macro avg       0.51      0.52      0.51      6000
weighted avg       0.56      0.57      0.56      6000



In [65]:
pickle.dump(model, open("final_social_media_model.pkl", "wb"))

In [66]:
pickle.dump(X.columns.tolist(), open("model_columns.pkl", "wb"))

In [23]:
print(y.value_counts())

target
low       15000
medium    15000
high      15000
Name: count, dtype: int64
